# 92. Scorecard Stability — Bootstrap Simulation

시계열 데이터 없이 **bootstrap 리샘플링**으로 scorecard 배점 안정성을 검증한다.

**실험 설계**:
1. 전체 train으로 reference surrogate 학습 → **bin 구조(edges) 확정**
2. Bootstrap 리샘플링 → surrogate 재학습 → **reference edges에 맞춰 score만 재계산**
3. bin 구조는 고정, score 변동만 측정 → 변동계수(CV)로 안정성 정량화

비교: 원본(full bins) vs cleansed(max_bins=10, min_ratio=1%)

**Note**: 본 노트북은 `logit(prob)` (높을수록 위험)을 사용한다. NB04-09는 `logit(1-prob)` (높을수록 안전)을 사용한다. CV 측정은 |mean| 기준이므로 부호 방향에 무관하다.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra._utils import logit
from decentra.surrogate import TreeSurrogate

DATA_DIR = '../.data'
N_BOOTSTRAP = 50
RS = 317

In [2]:
with open(f'{DATA_DIR}/base_models.pkl', 'rb') as f:
    base_models = pickle.load(f)

targets = {}
for name in base_models:
    bm = base_models[name]['lgb']
    s = base_models[name]['splits']
    targets[name] = {
        'y_logit_tr': logit(bm.predict_proba(s['X_tr'])[:, 1]),
        'y_logit_val': logit(bm.predict_proba(s['X_val'])[:, 1]),
        'y_logit_test': logit(bm.predict_proba(s['X_test'])[:, 1]),
    }

In [3]:
def evaluate_surrogate(surr, X_test, y_logit_test):
    pred = surr.predict(X_test)
    r2 = r2_score(y_logit_test, pred)
    sp = spearmanr(y_logit_test, pred)[0]
    mae = np.mean(np.abs(y_logit_test - pred))
    rmse = np.sqrt(np.mean((y_logit_test - pred) ** 2))
    return {'R2': round(r2, 4), 'Spearman': round(sp, 4),
            'MAE': round(mae, 4), 'RMSE': round(rmse, 4)}

---
## 1. Surrogate 성능 비교 (Tree depth=1/3/6)

In [4]:
for data_flag in ['GMSC', 'HC']:
    s = base_models[data_flag]['splits']
    t = targets[data_flag]
    X_tr, X_val, X_test = s['X_tr'], s['X_val'], s['X_test']
    y_tr, y_val, y_test = t['y_logit_tr'], t['y_logit_val'], t['y_logit_test']

    print(f'\n{"#"*80}')
    print(f'  {data_flag}')
    print(f'{"#"*80}')

    for d in [1, 3, 6]:
        print(f'\n{"="*80}')

        surr = TreeSurrogate(max_depth=d, verbose=0)
        surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        print(f'Tree depth={d}')
        print(evaluate_surrogate(surr, X_test, y_test))

        print('-' * 80)
        surr = TreeSurrogate(max_depth=d, monotone_detect_mode='auto', verbose=0)
        surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        print(f'Tree depth={d} + monotone')
        print(evaluate_surrogate(surr, X_test, y_test))

        if d == 1:
            print('-' * 80)
            surr = TreeSurrogate(max_depth=d, monotone_detect_mode='auto', verbose=0)
            surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            sm = surr.to_scorecard_model(X_tr, feature_names=list(X_tr.columns))
            print(f'Tree depth={d} + monotone -> ScorecardModel')
            print(evaluate_surrogate(sm, X_test, y_test))

            print('-' * 80)
            sm_clean = surr.to_scorecard_model(
                X_tr, feature_names=list(X_tr.columns),
                max_bins_per_feature=10, min_bin_ratio=0.01)
            print(f'Tree depth={d} + monotone -> ScorecardModel (bin clean)')
            print(evaluate_surrogate(sm_clean, X_test, y_test))


################################################################################
  GMSC
################################################################################



Tree depth=1
{'R2': 0.9403, 'Spearman': np.float64(0.9738), 'MAE': np.float64(0.2336), 'RMSE': np.float64(0.3253)}
--------------------------------------------------------------------------------


Tree depth=1 + monotone
{'R2': 0.9332, 'Spearman': np.float64(0.969), 'MAE': np.float64(0.2479), 'RMSE': np.float64(0.344)}
--------------------------------------------------------------------------------


Tree depth=1 + monotone -> ScorecardModel
{'R2': 0.9332, 'Spearman': np.float64(0.969), 'MAE': np.float64(0.2479), 'RMSE': np.float64(0.344)}
--------------------------------------------------------------------------------


Tree depth=1 + monotone -> ScorecardModel (bin clean)
{'R2': 0.932, 'Spearman': np.float64(0.9683), 'MAE': np.float64(0.2509), 'RMSE': np.float64(0.3472)}



...

Tree depth=1 + monotone -> ScorecardModel (bin clean)
{'R2': 0.8721, 'Spearman': np.float64(0.9369), 'MAE': np.float64(0.2336), 'RMSE': np.float64(0.303)}



Tree depth=3
{'R2': 0.9584, 'Spearman': np.float64(0.9793), 'MAE': np.float64(0.1325), 'RMSE': np.float64(0.1729)}
--------------------------------------------------------------------------------


Tree depth=3 + monotone
{'R2': 0.8954, 'Spearman': np.float64(0.9484), 'MAE': np.float64(0.2092), 'RMSE': np.float64(0.274)}



Tree depth=6
{'R2': 0.9768, 'Spearman': np.float64(0.9882), 'MAE': np.float64(0.0985), 'RMSE': np.float64(0.129)}
--------------------------------------------------------------------------------


Tree depth=6 + monotone
{'R2': 0.8965, 'Spearman': np.float64(0.95), 'MAE': np.float64(0.2075), 'RMSE': np.float64(0.2725)}


---
## 2. Bootstrap Stability Simulation

**설계**:
1. Reference: 전체 train으로 surrogate 학습 → ScorecardModel의 bin edges 확정
2. Bootstrap i: 리샘플링 → surrogate 재학습 → **reference edges 기준**으로 각 bin의 mean contribution 재계산
3. bin 구조 동일 → score 변동만 순수 측정

In [5]:
def bootstrap_stability(X_tr, y_tr, X_val, y_val, feature_names,
                         n_bootstrap=50, max_bins=None, min_ratio=0.0,
                         random_state=317):
    """Bootstrap surrogate → measure score variability on fixed reference bins.
    
    Step 1: Fit reference surrogate on full train → fix bin edges.
    Step 2: For each bootstrap, fit new surrogate, compute mean
            contribution per reference bin.
    
    Returns
    -------
    ref_info : list of dict
        [{feature, bin_idx, lower, upper, ref_score}, ...]
    boot_scores : ndarray of shape (n_bootstrap, n_total_bins)
        Score for each bin at each bootstrap iteration.
    """
    rng = np.random.RandomState(random_state)
    n = len(X_tr)
    X_arr = np.asarray(X_tr)
    
    # Step 1: Reference model → fixed bin structure
    surr_ref = TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0)
    surr_ref.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    sm_ref = surr_ref.to_scorecard_model(
        X_tr, feature_names=feature_names,
        max_bins_per_feature=max_bins, min_bin_ratio=min_ratio)
    
    # Flatten reference bins
    ref_info = []
    for feat in sm_ref.features:
        for bi, br in enumerate(feat.bins):
            ref_info.append({
                'feature': feat.name,
                'feature_idx': feat.index,
                'bin_idx': bi,
                'lower': br.lower,
                'upper': br.upper,
                'ref_score': br.score,
            })
    
    # Pre-compute reference bin masks on X_tr (for assigning bootstrap contributions)
    bin_masks = []
    for r in ref_info:
        j = r['feature_idx']
        mask = (X_arr[:, j] >= r['lower']) & (X_arr[:, j] < r['upper'])
        bin_masks.append(mask)
    
    # Step 2: Bootstrap
    boot_scores = np.full((n_bootstrap, len(ref_info)), np.nan)
    
    for i in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        if hasattr(X_tr, 'iloc'):
            X_b = X_tr.iloc[idx].reset_index(drop=True)
        else:
            X_b = X_tr[idx]
        y_b = y_tr[idx]
        
        surr_b = TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0)
        surr_b.fit(X_b, y_b, eval_set=(X_val, y_val))
        
        # Compute contributions on FULL X_tr (not bootstrap sample)
        contribs_b = surr_b.contributions(X_tr)
        
        # Assign mean contribution per reference bin
        for k, r in enumerate(ref_info):
            j = r['feature_idx']
            mask = bin_masks[k]
            if mask.sum() > 0:
                boot_scores[i, k] = np.mean(contribs_b[mask, j])
    
    return ref_info, boot_scores

In [6]:
def stability_summary(ref_info, boot_scores):
    """Compute stability metrics per bin."""
    rows = []
    for k, r in enumerate(ref_info):
        scores = boot_scores[:, k]
        valid = scores[~np.isnan(scores)]
        if len(valid) < 10:
            continue
        
        mean_s = np.mean(valid)
        std_s = np.std(valid)
        cv = std_s / np.abs(mean_s) if np.abs(mean_s) > 1e-10 else np.nan
        
        lo, hi = r['lower'], r['upper']
        if np.isinf(lo) and lo < 0:
            label = f'< {hi:.4g}'
        elif np.isinf(hi):
            label = f'>= {lo:.4g}'
        else:
            label = f'{lo:.4g} ~ {hi:.4g}'
        
        rows.append({
            'Feature': r['feature'],
            'Bin': label,
            'Ref Score': round(r['ref_score'], 4),
            'Boot Mean': round(mean_s, 4),
            'Boot Std': round(std_s, 4),
            'CV': round(cv, 4) if not np.isnan(cv) else np.nan,
        })
    
    return pd.DataFrame(rows)

### 2-1. GMSC

In [7]:
data_flag = 'GMSC'
s = base_models[data_flag]['splits']
t = targets[data_flag]
fnames = list(s['X_tr'].columns)

print(f'=== {data_flag}: Bootstrap {N_BOOTSTRAP}회 ===')

t0 = time.time()
ref_orig, boot_orig = bootstrap_stability(
    s['X_tr'], t['y_logit_tr'], s['X_val'], t['y_logit_val'],
    fnames, n_bootstrap=N_BOOTSTRAP)
print(f'Original (full bins): {len(ref_orig)} bins, {time.time()-t0:.0f}s')

t0 = time.time()
ref_clean, boot_clean = bootstrap_stability(
    s['X_tr'], t['y_logit_tr'], s['X_val'], t['y_logit_val'],
    fnames, n_bootstrap=N_BOOTSTRAP, max_bins=10, min_ratio=0.01)
print(f'Cleansed (max=10, 1%): {len(ref_clean)} bins, {time.time()-t0:.0f}s')

=== GMSC: Bootstrap 50회 ===


Original (full bins): 143 bins, 48s


Cleansed (max=10, 1%): 65 bins, 47s


In [8]:
df_orig = stability_summary(ref_orig, boot_orig)
df_clean = stability_summary(ref_clean, boot_clean)

print(f'=== {data_flag} — Original ({len(df_orig)} bins) ===')
print(f'  Mean CV: {df_orig["CV"].mean():.4f}')
print(f'  Max CV:  {df_orig["CV"].max():.4f}')
print(f'  Bins with CV > 0.5: {(df_orig["CV"] > 0.5).sum()}')

print(f'\n=== {data_flag} — Cleansed ({len(df_clean)} bins) ===')
print(f'  Mean CV: {df_clean["CV"].mean():.4f}')
print(f'  Max CV:  {df_clean["CV"].max():.4f}')
print(f'  Bins with CV > 0.5: {(df_clean["CV"] > 0.5).sum()}')

=== GMSC — Original (143 bins) ===
  Mean CV: 0.1034
  Max CV:  1.5393
  Bins with CV > 0.5: 7

=== GMSC — Cleansed (65 bins) ===
  Mean CV: 0.0913
  Max CV:  1.5136
  Bins with CV > 0.5: 3


In [9]:
print(f'=== {data_flag} — Original: 가장 불안정한 bins (CV 상위 10) ===')
print(df_orig.sort_values('CV', ascending=False).head(10).to_string(index=False))

print(f'\n=== {data_flag} — Cleansed: 가장 불안정한 bins (CV 상위 10) ===')
print(df_clean.sort_values('CV', ascending=False).head(10).to_string(index=False))

=== GMSC — Original: 가장 불안정한 bins (CV 상위 10) ===
                             Feature             Bin  Ref Score  Boot Mean  Boot Std     CV
RevolvingUtilizationOfUnsecuredLines  0.295 ~ 0.3001     0.0317     0.0215    0.0332 1.5393
        NumberRealEstateLoansOrLines       1.5 ~ 2.5     0.0009     0.0012    0.0018 1.5136
                           DebtRatio 0.3492 ~ 0.4227    -0.0046    -0.0028    0.0033 1.1881
                                 age     54.5 ~ 55.5    -0.0196    -0.0156    0.0116 0.7476
                           DebtRatio 0.4227 ~ 0.4701     0.0056     0.0064    0.0046 0.7087
                       MonthlyIncome     5301 ~ 5337     0.0365     0.0292    0.0163 0.5576
                           DebtRatio 0.3241 ~ 0.3492    -0.0107    -0.0107    0.0055 0.5097
                           DebtRatio 0.4701 ~ 0.4792     0.0198     0.0151    0.0064 0.4259
                       MonthlyIncome     5337 ~ 5387    -0.0399    -0.0409    0.0158 0.3865
                           Debt

### 2-2. HC

In [10]:
data_flag = 'HC'
s = base_models[data_flag]['splits']
t = targets[data_flag]
fnames = list(s['X_tr'].columns)

print(f'=== {data_flag}: Bootstrap {N_BOOTSTRAP}회 ===')

t0 = time.time()
ref_orig_hc, boot_orig_hc = bootstrap_stability(
    s['X_tr'], t['y_logit_tr'], s['X_val'], t['y_logit_val'],
    fnames, n_bootstrap=N_BOOTSTRAP)
print(f'Original (full bins): {len(ref_orig_hc)} bins, {time.time()-t0:.0f}s')

t0 = time.time()
ref_clean_hc, boot_clean_hc = bootstrap_stability(
    s['X_tr'], t['y_logit_tr'], s['X_val'], t['y_logit_val'],
    fnames, n_bootstrap=N_BOOTSTRAP, max_bins=10, min_ratio=0.01)
print(f'Cleansed (max=10, 1%): {len(ref_clean_hc)} bins, {time.time()-t0:.0f}s')

=== HC: Bootstrap 50회 ===


Original (full bins): 291 bins, 191s


Cleansed (max=10, 1%): 133 bins, 188s


In [11]:
df_orig_hc = stability_summary(ref_orig_hc, boot_orig_hc)
df_clean_hc = stability_summary(ref_clean_hc, boot_clean_hc)

print(f'=== HC — Original ({len(df_orig_hc)} bins) ===')
print(f'  Mean CV: {df_orig_hc["CV"].mean():.4f}')
print(f'  Max CV:  {df_orig_hc["CV"].max():.4f}')
print(f'  Bins with CV > 0.5: {(df_orig_hc["CV"] > 0.5).sum()}')

print(f'\n=== HC — Cleansed ({len(df_clean_hc)} bins) ===')
print(f'  Mean CV: {df_clean_hc["CV"].mean():.4f}')
print(f'  Max CV:  {df_clean_hc["CV"].max():.4f}')
print(f'  Bins with CV > 0.5: {(df_clean_hc["CV"] > 0.5).sum()}')

=== HC — Original (291 bins) ===
  Mean CV: 0.1800
  Max CV:  12.2218
  Bins with CV > 0.5: 18

=== HC — Cleansed (133 bins) ===
  Mean CV: 0.2086
  Max CV:  4.8622
  Bins with CV > 0.5: 10


---
## 3. Summary

In [12]:
summary = pd.DataFrame([
    {'Dataset': 'GMSC', 'Type': 'Original',
     'Total Bins': len(df_orig), 'Mean CV': df_orig['CV'].mean(),
     'Max CV': df_orig['CV'].max(), 'Bins CV>0.5': int((df_orig['CV'] > 0.5).sum())},
    {'Dataset': 'GMSC', 'Type': 'Cleansed',
     'Total Bins': len(df_clean), 'Mean CV': df_clean['CV'].mean(),
     'Max CV': df_clean['CV'].max(), 'Bins CV>0.5': int((df_clean['CV'] > 0.5).sum())},
    {'Dataset': 'HC', 'Type': 'Original',
     'Total Bins': len(df_orig_hc), 'Mean CV': df_orig_hc['CV'].mean(),
     'Max CV': df_orig_hc['CV'].max(), 'Bins CV>0.5': int((df_orig_hc['CV'] > 0.5).sum())},
    {'Dataset': 'HC', 'Type': 'Cleansed',
     'Total Bins': len(df_clean_hc), 'Mean CV': df_clean_hc['CV'].mean(),
     'Max CV': df_clean_hc['CV'].max(), 'Bins CV>0.5': int((df_clean_hc['CV'] > 0.5).sum())},
]).round(4)

print(f'=== Bootstrap Stability Summary (N={N_BOOTSTRAP}) ===')
summary

=== Bootstrap Stability Summary (N=50) ===


,Dataset,Type,Total Bins,Mean CV,Max CV,Bins CV>0.5
0,GMSC,Original,143,0.1034,1.5393,7
1,GMSC,Cleansed,65,0.0913,1.5136,3
2,HC,Original,291,0.1800,12.2218,18
3,HC,Cleansed,133,0.2086,4.8622,10
